# Class Notes → MCQ Question Generator

Generates 4 multiple-choice questions (4 options, exactly 1 correct) from class notes.

Here user can choose anyone LLM provider: Gemini / OpenAI / Cohere. It is a one-line change in the config cell.

Uses LangChain's `init_chat_model`, so no provider-specific code elsewhere.

## 1. Install

In [ ]:
%pip install -q langchain==1.3.6 langchain-google-genai==4.4.0 langchain-openai==1.2.2 langchain-cohere==0.6.0 pydantic python-dotenv

## 2. Imports and API keys

`.env` file next to the notebook (only the key for the provider you use is required):

```text
GOOGLE_API_KEY=...
OPENAI_API_KEY=...
COHERE_API_KEY=...
```

In [1]:
import os
import json
import random
from typing import List

from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator, model_validator

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

True

## 3. Config: switch provider here

Set `PROVIDER` to `"gemini"`, `"openai"`, or `"cohere"`. Edit `MODELS` if you want a different model for a provider.

In [2]:
PROVIDER = "gemini"          # Here OPTIONS = "gemini" | "openai" | "cohere"

NUM_QUESTIONS = 4
TEMPERATURE = 0.3
MAX_RETRIES = 3

MODELS = {
    "gemini": "gemini-2.5-flash",
    "openai": "gpt-4o-mini",
    "cohere": "command-a-03-2025",
}

# provider key -> (init_chat_model provider id, required env var)
PROVIDER_INFO = {
    "gemini": ("google_genai", "GOOGLE_API_KEY"),
    "openai": ("openai", "OPENAI_API_KEY"),
    "cohere": ("cohere", "COHERE_API_KEY"),
}

## 4. Build the model

In [3]:
def build_llm(provider: str):
    if provider not in PROVIDER_INFO:
        raise ValueError(f"Unknown provider '{provider}'. Choose from {list(PROVIDER_INFO)}")

    model_provider, env_var = PROVIDER_INFO[provider]

    if not os.getenv(env_var):
        raise ValueError(f"{env_var} not found. Add it to your .env file.")

    model_name = MODELS[provider]
    kwargs = {}

    # OpenAI reasoning models (gpt-5*, o*) reject a custom temperature
    is_openai_reasoning = provider == "openai" and model_name.startswith(("gpt-5", "o1", "o3", "o4"))
    if not is_openai_reasoning:
        kwargs["temperature"] = TEMPERATURE

    return init_chat_model(model=model_name, model_provider=model_provider, **kwargs)


llm = build_llm(PROVIDER)
print(f"Provider: {PROVIDER} | Model: {MODELS[PROVIDER]}")

Provider: gemini | Model: gemini-2.5-flash


## 5. Output schema

Validators enforce: 4 options, all distinct, a valid correct index.

In [4]:
class MCQ(BaseModel):
    question: str = Field(description="The question text")
    
    options: List[str] = Field(
        description="Exactly 4 distinct answer options, without letter prefixes like 'A)'",
        min_length=4,
        max_length=4,
    )
    
    correct_index: int = Field(
        description="0-based index (0-3) of the single correct option",
        ge=0,
        le=3,
    )
    
    explanation: str = Field(description="Brief explanation of why the correct option is right")

    @field_validator("options")
    @classmethod
    def options_distinct(cls, v):
        if len({o.strip().lower() for o in v}) != len(v):
            raise ValueError("Options must be distinct")
        return v


class Quiz(BaseModel):
    questions: List[MCQ] = Field(description="The list of multiple-choice questions")

## 6. Prompt and chain

In [5]:
system_prompt = """
    You are an experienced teacher writing an assessment from class notes.
    
    Rules:
    - Write exactly {num_questions} multiple-choice questions.
    - Base every question strictly on the provided notes. Do not use outside knowledge.
    - Each question has exactly 4 options and exactly ONE correct option.
    - The 3 wrong options must be plausible but clearly incorrect according to the notes.
    - Cover different parts of the notes; do not repeat the same concept.
    - Do not use "all of the above" or "none of the above".
    - Do not prefix options with letters or numbers.
    - Do not refer to options by position in the explanation.
    - Keep questions clear and unambiguous.
"""

human_prompt = """
    Class notes:
    {notes}
    
    Generate {num_questions} multiple-choice questions.
"""

prompt = ChatPromptTemplate.from_messages([
    ( "system",system_prompt ,),
    ( "human",human_prompt,),
])

structured_llm = llm.with_structured_output(Quiz)
chain = prompt | structured_llm

## 7. Generation function

Retries on malformed output. Options are shuffled in code, since LLMs tend to bias the correct answer to certain positions.

In [6]:
def shuffle_options(q: MCQ) -> MCQ:
    correct_text = q.options[q.correct_index]
    options = q.options[:]
    random.shuffle(options)
    return q.model_copy(update={
        "options": options,
        "correct_index": options.index(correct_text),
    })


def generate_quiz(notes: str, num_questions: int = NUM_QUESTIONS) -> Quiz:
    if not notes.strip():
        raise ValueError("Notes are empty.")

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = chain.invoke({"notes": notes, "num_questions": num_questions})

            if isinstance(result, dict):
                result = Quiz.model_validate(result)

            if len(result.questions) != num_questions:
                raise ValueError(f"Expected {num_questions} questions, got {len(result.questions)}")

            return Quiz(questions=[shuffle_options(q) for q in result.questions])

        except Exception as e:
            last_error = e
            print(f"Attempt {attempt} failed: {e}")

    raise RuntimeError(f"Quiz generation failed after {MAX_RETRIES} attempts: {last_error}")

## 8. Display and export

In [7]:
LETTERS = "ABCD"


def display_quiz(quiz: Quiz, show_answers: bool = True) -> None:
    for i, q in enumerate(quiz.questions, start=1):
        print(f"\nQ{i}. {q.question}")
        for letter, option in zip(LETTERS, q.options):
            print(f"   {letter}) {option}")

    if show_answers:
        print("\n" + "=" * 60)
        print("ANSWER KEY")
        print("=" * 60)
        for i, q in enumerate(quiz.questions, start=1):
            print(f"Q{i}. {LETTERS[q.correct_index]}) {q.options[q.correct_index]}")
            print(f"    {q.explanation}")


def save_quiz(quiz: Quiz, path: str = "quiz.json") -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(quiz.model_dump(), f, indent=2, ensure_ascii=False)
    print(f"Saved to {path}")

## 9. Add your class notes

Paste notes below, or load them from a `.txt` / `.md` file (see the next cell).

In [8]:
NOTES = """
Photosynthesis
- Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy stored in glucose.
- It occurs in chloroplasts, which contain the pigment chlorophyll. Chlorophyll absorbs mostly red and blue light and reflects green light.
- Light-dependent reactions occur in the thylakoid membranes. They split water, releasing oxygen, and produce ATP and NADPH.
- The Calvin cycle (light-independent reactions) occurs in the stroma. It uses ATP and NADPH to fix carbon dioxide into glucose.
- Overall equation: 6CO2 + 6H2O + light energy -> C6H12O6 + 6O2.
- Factors affecting the rate: light intensity, carbon dioxide concentration, and temperature.
"""

In [ ]:
# Optional: load notes from a file instead
# with open("class_notes.txt", encoding="utf-8") as f:
#     NOTES = f.read()

## 10. Generate

In [9]:
quiz = generate_quiz(NOTES)
display_quiz(quiz)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



Q1. What is the primary outcome of photosynthesis?
   A) Conversion of light energy into chemical energy stored in glucose.
   B) Production of carbon dioxide and water.
   C) Breakdown of glucose to release light energy.
   D) Conversion of chemical energy into light energy.

Q2. Where do the light-dependent reactions of photosynthesis take place?
   A) In the cytoplasm.
   B) In the thylakoid membranes.
   C) In the stroma.
   D) In the nucleus.

Q3. Which statement accurately describes the light absorption properties of chlorophyll?
   A) It absorbs mostly green light.
   B) It reflects mostly red and blue light.
   C) It absorbs all wavelengths of light equally.
   D) It absorbs mostly red and blue light.

Q4. What are the direct products released or produced during the light-dependent reactions?
   A) Carbon dioxide and ATP.
   B) ATP, NADPH, and oxygen.
   C) Water and glucose.
   D) Glucose and carbon dioxide.

ANSWER KEY
Q1. A) Conversion of light energy into chemical energy s

In [ ]:
# Optional: questions only (no answers), and save to JSON
# display_quiz(quiz, show_answers=False)
# save_quiz(quiz, "quiz.json")

## Switching providers

Change one line in the config cell and re-run from there:

```python
PROVIDER = "openai"   # or "cohere" or "gemini"
```

To add a new provider, add one entry each to `MODELS` and `PROVIDER_INFO` (any provider supported by `init_chat_model`).

# STOP

# Further enhancement

**Easy**
- Difficulty levels — add a `difficulty: Literal["easy","medium","hard"]` field to `MCQ` and let the prompt vary it, or take `DIFFICULTY` as a config input.
- Configurable option count — generalize beyond exactly 4 options (some instructors want 3 or 5).
- Export formats — write to CSV, or to a format importable by Google Forms / Kahoot / Quizlet, not just JSON.
- Answer sheet vs. quiz sheet — separate printable/exportable files: one with just questions, one with the answer key.
- Topic tagging — add a `topic` field per question so questions can be filtered/grouped by subtopic later.

**Medium**
- Duplicate/overlap detection across questions — right now nothing stops the LLM from asking two questions about the same fact; could embed each question (via the same provider's embedding model) and flag high cosine-similarity pairs.
- Bloom's taxonomy control — tag or request questions at specific cognitive levels (recall, application, analysis) instead of leaving that to chance.
- Distractor quality check — a second LLM pass that scores each wrong option on "plausibility" and regenerates weak ones (e.g., options that are obviously wrong or trivially eliminable).
- Source grounding / citations — have the model quote or point to the specific line/sentence in the notes that each question is based on, so a teacher can verify it against the source instead of trusting it blindly.
- Student-facing quiz mode — an interactive loop (`input()` in notebook, or a small Gradio form) that presents questions one at a time, takes answers, and scores the student — turning it from a generator into an actual quiz-taking tool.

**Harder**
- Chunking for long notes — same issue as the meeting-minutes notebook: right now a full textbook chapter would overflow context. Chunk notes, generate questions per chunk, then dedupe.
- Multi-provider ensembling — generate the same question set from two providers and cross-check for factual disagreement, useful as a validation step against hallucinated distractors.
- Adaptive difficulty — track which questions students get wrong across sessions and bias future generation toward weaker topics (needs persistent storage, e.g. a small SQLite file).
- Image/diagram-based questions — if notes include diagrams (e.g. a labeled biology figure), use a multimodal model to generate questions that reference the image.
- Full item-analysis pipeline — after a cohort takes the quiz, compute per-question discrimination index and difficulty index (standard psychometrics) to flag bad questions automatically.
